# Data Cleaning

Load the full merged item table, inspect input integrity, run the reusable cleaning workflow, and review its change and quality summaries. Validate the 28-column clean result, then explicitly save and reload `cleaned_order_items` in `data/processed`. The workflow retains every item and order status.


## 1. Locate the project

Run the notebook from the repository root or from `notebooks`. The root search also supports a working directory nested inside the repository.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    """Find the repository containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError(
        "Could not locate the project root containing src/ and requirements.txt."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.cleaning import (
    CLEANED_DATASET_NAME,
    assert_cleaned_roundtrip,
    clean_order_items,
    format_cleaning_preview,
    inspect_text_fields,
    save_cleaned_data,
)
from src.cleaning.validation import (
    assert_input_unchanged,
    capture_input_baseline,
    validate_cleaning_input,
)
from src.data_loader import load_data
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA
from src.paths import INTERIM_DIR, PROCESSED_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data path: {INTERIM_DIR}")
print(f"Processed data path: {PROCESSED_DIR}")


## 2. Load the full merged table

Load `merged_order_items` from `data/interim` through the schema-aware project loader. Parquet is preferred, with CSV fallback; use `file_format="csv"` to select CSV directly.

In [ ]:
df = load_data("merged_order_items", stage="interim")

## 3. Capture the input baseline

Keep the baseline in memory. It records dimensions, column order, dtypes and row hashes that include the index. No second full-table copy or baseline file is created.

In [ ]:
input_baseline = capture_input_baseline(df)
input_summary = pd.DataFrame(
    [
        ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
    ],
    columns=["Metric", "Reference", "Observed"],
)
input_summary["Matches reference"] = input_summary["Reference"].eq(
    input_summary["Observed"]
)
display(format_cleaning_preview(input_summary))

## 4. Validate the cleaning input

Check exact columns and order, canonical dtypes, mandatory non-null identifiers, the unique item key and consistent repeated attributes. Repeated order, product, seller and customer IDs are expected at item grain.

Inspect string IDs as 32 lowercase hexadecimal characters, ZIP prefixes as five ASCII digits, and state/status values against fixed source domains. The summary separates missing, valid and invalid observed item values. Missing optional attributes are retained; invalid observed text values stop input validation. The full `text_inspection["issues"]` table records original values and item positions, including duplicate index labels.


In [ ]:
text_inspection = inspect_text_fields(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Text formats and source domains | item grain")
    display(format_cleaning_preview(text_inspection["summary"]))
    if not text_inspection["issues"].empty:
        print("First ten invalid text cells; full records: text_inspection['issues']")
        display(format_cleaning_preview(text_inspection["issues"]))

input_validation = validate_cleaning_input(df)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("Key profiles")
    display(format_cleaning_preview(input_validation["key_profile"]))
    print("Entity consistency")
    display(format_cleaning_preview(input_validation["entity_consistency"]))
    if not input_validation["conflicts"].empty:
        display(format_cleaning_preview(input_validation["conflicts"]))


## 5. Confirm the input baseline

Compare the loaded table against its baseline before running the transformations.


In [ ]:
assert_input_unchanged(df, input_baseline)
print("Input validated and unchanged.")

## 6. Run cleaning and review text changes

Run `clean_order_items(df)` once. It validates input, normalizes city whitespace, handles non-positive physical measurements and adds temporal quality flags. The returned `cleaning_details` tables support the inspections below. Before returning, the function verifies the clean schema, every permitted change and source preservation.

City normalization trims the edges of `customer_city` and `seller_city` and replaces internal whitespace runs with one regular space. Its summary counts changed items and entities.


In [ ]:
cleaned_df, cleaning_details = clean_order_items(df)
text_change_summary = cleaning_details["text_changes"]
display(format_cleaning_preview(text_change_summary))


## 7. Handle non-positive physical measurements

Set observed weight or dimensions at or below zero to missing. Keep pre-existing missing measurements, freight values, and numeric tails unchanged. Flag affected item rows so downstream work can distinguish newly invalidated product measurements from values that were already missing.

In [ ]:
measurement_summary = cleaning_details["measurements"]
display(format_cleaning_preview(measurement_summary))
print("Item rows flagged:", int(cleaned_df["has_invalid_physical_measurement"].sum()))


## 8. Record timestamp ordering and deadline review signals

Check the adjacent sequence Purchase → Approved → Carrier Handoff → Customer Delivery once per order. Preserve original timestamps. A comparison with a missing endpoint remains unknown; missing core timestamps are summarized separately. Flag affected orders on their item rows for later eligibility checks.

At item grain, flag deadlines before purchase and deadlines whose year exceeds the latest observed purchase year in the full input. The latter is an observation-based review signal, not a delivery SLA or a definitive invalid-date rule. `needs_deadline_review` is true when either check is observed true. False means no observed issue; check the unknown counts for missing comparisons. The full `deadline_review_records` table retains original item records and both nullable check results.


In [ ]:
timeline_summary = cleaning_details["timeline"]
core_timestamp_summary = cleaning_details["core_timestamps"]
deadline_summary = cleaning_details["deadlines"]
deadline_review_records = cleaning_details["deadline_records"]

display(format_cleaning_preview(timeline_summary))
display(format_cleaning_preview(core_timestamp_summary))
print("Item rows flagged for timestamp ordering:", int(cleaned_df["has_invalid_timeline"].sum()))
display(format_cleaning_preview(deadline_summary))
if not deadline_review_records.empty:
    print("First ten deadline-review items; full records: deadline_review_records")
    with pd.option_context("display.max_columns", None):
        display(format_cleaning_preview(deadline_review_records))
print("Item rows flagged for deadline review:", int(cleaned_df["needs_deadline_review"].sum()))


## 9. Review a formatted cleaning preview

Show up to ten cleaned item rows with readable column labels, units and display values. The preview is a separate presentation table; `cleaned_df` retains its numeric, timestamp, boolean, ZIP and identifier types.

In [ ]:
cleaning_preview = format_cleaning_preview(cleaned_df, rows=10)
display(cleaning_preview)


## 10. Review output validation

Review the successful schema, item-key, allowed-change and quality-flag checks from the workflow. Confirm the loaded interim input still matches its original baseline.


In [ ]:
display(format_cleaning_preview(cleaning_details["output_checks"]))
assert_input_unchanged(df, input_baseline)
print("Clean result validated; loaded interim input remains unchanged.")


## 11. Save and verify the processed dataset

Save with the fixed clean schema through the shared atomic Parquet/CSV fallback saver. Read back the actual successful format so verification targets this result even if another format already exists. The comparison checks all values, dtypes and row order. File storage preserves item keys, not pandas index labels.

For later independent reads, `load_data("cleaned_order_items", stage="processed")` prefers Parquet with CSV fallback. Add `file_format="csv"` to select CSV directly.


In [ ]:
saved_path = save_cleaned_data(cleaned_df)
reloaded_df = load_data(
    CLEANED_DATASET_NAME,
    stage="processed",
    file_format=saved_path.suffix.lstrip("."),
)
assert_cleaned_roundtrip(cleaned_df, reloaded_df)
print(f"Verified processed output: {saved_path}")
